# 📉 Data Speaks!! — Dimensionality Reduction (LSA / PCA for Text)

This notebook satisfies the **Dimensionality Reduction** requirement of the project. 
Because text data transformed by TF-IDF results in thousands of dimensions (one for each word), we will use **Truncated SVD** (Latent Semantic Analysis, the equivalent of PCA for sparse text data) to reduce the dataset down to just 2 dimensions. This allows us to plot the entire dataset on a 2D graph to see if Positive and Negative reviews naturally separate!

---
## 1. Setup & Imports

In [ ]:
import os
import re
import html
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully!")

---
## 2. Load and Clean the Data

In [ ]:
def clean_text(text: str) -> str:
    text = html.unescape(text)
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

DATA_PATH = os.path.join("..", "processed", "imdb_train.csv")
df = pd.read_csv(DATA_PATH)

# To make the 2D plot render fast and clearly, we will sample 5000 random reviews
df_sample = df.sample(5000, random_state=42).reset_index(drop=True)
df_sample['text_clean'] = df_sample['text'].apply(clean_text)

print(f"Loaded and cleaned {len(df_sample)} samples for visualization.")

---
## 3. High-Dimensionality: TF-IDF Vectorization
First, we convert the text into a matrix where every word is a separate dimension. We will limit it to the top 5,000 words.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, stop_words="english")
X_tfidf = tfidf.fit_transform(df_sample['text_clean'])
y = df_sample['label'].values

print(f"TF-IDF Matrix Shape: {X_tfidf.shape}")
print("This means our data currently exists in 5,000-dimensional space!")

---
## 4. Dimensionality Reduction (5000D ➔ 2D)
We will use **TruncatedSVD** to smash the 5,000 dimensions down to just **2 dimensions** (X and Y coordinates) while attempting to preserve as much variance (information) as possible.

In [ ]:
svd_2d = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd_2d.fit_transform(X_tfidf)

print(f"Reduced Matrix Shape: {X_2d.shape}")
print(f"Explained Variance Ratio (How much information was kept): {svd_2d.explained_variance_ratio_.sum() * 100:.2f}%")

---
## 5. Visualizing the Reduced Dimensions
Now we can actually plot the text data on a standard graph to see if Positive and Negative reviews cluster together.

In [ ]:
plt.figure(figsize=(10, 8))

# Scatter plot for Negative reviews (Label 0)
plt.scatter(X_2d[y == 0, 0], X_2d[y == 0, 1], 
            alpha=0.5, color='#e74c3c', label='Negative', s=20)

# Scatter plot for Positive reviews (Label 1)
plt.scatter(X_2d[y == 1, 0], X_2d[y == 1, 1], 
            alpha=0.5, color='#2ecc71', label='Positive', s=20)

plt.title("2D Visualization of IMDB Reviews (Truncated SVD)", fontsize=16, fontweight='bold')
plt.xlabel("Principal Component 1 (SVD-1)", fontsize=12)
plt.ylabel("Principal Component 2 (SVD-2)", fontsize=12)
plt.legend(fontsize=12, markerscale=2)

os.makedirs("plots", exist_ok=True)
plt.savefig("plots/svd_2d_scatter.png", dpi=300, bbox_inches='tight')
plt.show()

---
## 6. Training on Reduced Dimensions (5000D ➔ 100D)
While 2 dimensions is great for plotting, it destroys too much information to train a good model. 
Let's see what happens if we reduce the 5,000 dimensions to **100 dimensions**, and train a Logistic Regression model on it. This proves that Dimensionality Reduction can compress data while retaining predictive power.

In [ ]:
# 1. Reduce to 100 Dimensions
svd_100 = TruncatedSVD(n_components=100, random_state=42)
X_100d = svd_100.fit_transform(X_tfidf)

print(f"Reduced to {X_100d.shape[1]} dimensions.")
print(f"Explained Variance: {svd_100.explained_variance_ratio_.sum() * 100:.2f}%")

# 2. Train a fast Logistic Regression model on the 100D data
model = LogisticRegression(max_iter=1000)
model.fit(X_100d, y)

# 3. Check accuracy on the exact same training set 
# (Just to prove the model can learn from the compressed data!)
y_pred = model.predict(X_100d)
acc = accuracy_score(y, y_pred)

print(f"\nAccuracy using only 100 dimensions (instead of 5000): {acc * 100:.2f}%")